# Iris Dataset — Data Preparation

This notebook demonstrates cleaning a deliberately messy practice copy of the public Iris dataset. The messy values were introduced for learning; they are **not** claims about the original public dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# If opened from the notebooks folder, project root is its parent.
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

raw_path = ROOT / "data" / "iris_raw_messy.csv"
df = pd.read_csv(raw_path, dtype=str)
df.head()


## 1. Understand the raw data

The dataset has four flower measurements in centimetres and one species label.

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.head(10))


## 2. Standardize missing-value markers

In [ ]:
missing_markers = ["", "NA", "N/A", "missing", "None", "null"]
df = df.replace(missing_markers, pd.NA)
df = df.replace(r"^\\s*$", pd.NA, regex=True)

print("Missing values by column:")
display(df.isna().sum())


## 3. Fix data types and invalid numeric values

Measurement columns should be numeric. Any value that cannot be parsed is treated as missing, rather than guessed.

In [ ]:
numeric_cols = [
    "sepal_length_cm", "sepal_width_cm",
    "petal_length_cm", "petal_width_cm"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype("string").str.strip(), errors="coerce")

print(df.dtypes)
display(df.isna().sum())


## 4. Standardize category labels

In [ ]:
df["species"] = (
    df["species"].astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "iris-setosa": "Iris-setosa",
        "iris-versicolor": "Iris-versicolor",
        "iris-virginica": "Iris-virginica"
    })
)
display(df["species"].value_counts(dropna=False))


## 5. Impute missing numeric values

**Assumption:** use each measurement column's median. Median imputation keeps all rows and is less affected by extreme values than mean imputation. In a real project, discuss this choice with your supervisor and investigate why values are missing.

In [ ]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df["species"] = df["species"].astype("category")

print("Missing values after cleaning:")
display(df.isna().sum())
print("Final shape:", df.shape)
display(df.describe(include="all"))


## 6. Basic quality checks

In [ ]:
assert df.isna().sum().sum() == 0
assert df["species"].isin(
    ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
).all()
assert (df[numeric_cols] > 0).all().all()

print("All validation checks passed.")


## 7. Quick visualization

In [ ]:
df["species"].value_counts().plot(kind="bar", title="Flower count by species")
plt.xlabel("Species")
plt.ylabel("Number of rows")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 8. Export cleaned dataset

In [ ]:
output_path = ROOT / "data" / "iris_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
